# Mattina 2 — Reti neurali su dati tabellari, fatte bene

**Corso «Deep Learning per il Rischio di Credito: stima della PD con PyTorch»** — notebook di laboratorio della seconda mattina.

Nella mattina 1 il nostro primo MLP (percettrone multistrato) e' rimasto dietro al champion XGBoost: era prevedibile, perche' lo avevamo schierato *disarmato*. Oggi lo equipaggiamo sul serio, su un dataset piu' difficile e realistico:

1. **preprocessing per reti**: valori mancanti, scaling e — soprattutto — **embedding** per le variabili categoriche ad alta cardinalita', al posto del one-hot;
2. **regolarizzazione e sbilanciamento**: dropout, batch normalization, weight decay, early stopping, con **esperimenti controllati** (una tecnica alla volta); poi pesi di classe e **focal loss**;
3. **calibrazione** dell'output della rete (Platt, isotonica, temperature scaling) e **verdetto champion/challenger** finale contro XGBoost, letto come lo leggerebbe un validatore.

**Come usare questo notebook in Visual Studio Code**: aprire il file, selezionare in alto a destra il kernel dell'ambiente conda del corso, ed eseguire le celle **in ordine, dall'alto verso il basso** (pulsante ▶ a sinistra di ogni cella, oppure `Maiusc+Invio`). Ogni cella di codice e' preceduta da una spiegazione: leggiamo prima, eseguiamo poi.

## Assunzioni di questo notebook

Esplicitiamo tutte le assunzioni, cosi' nessun risultato va interpretato oltre il suo perimetro.

| # | Assunzione |
|---|-----------|
| 1 | **Dataset**: campione del [*Home Credit Default Risk*](https://www.kaggle.com/competitions/home-credit-default-risk/data?select=application_train.csv) (Kaggle, 2018) — 80.000 domande di prestito al consumo, 31 colonne selezionate dalle 122 originali. Il file e' fornito col corso in `dati/home_credit_campione.csv`. In sua assenza il notebook usa, se presente, il segnaposto sintetico `dati/home_credit_sintetico.csv`, che replica lo schema del campione e serve solo a far girare il codice |
| 2 | **Target**: la colonna `TARGET` vale 1 se il cliente ha avuto **difficolta' di pagamento** (ritardi rilevanti sulle prime rate del prestito). Come per il Taiwan della mattina 1, la usiamo come **proxy didattico** della PD: nel mondo reale il riferimento e' la definizione regolamentare (art. 178 CRR) con orizzonte a 12 mesi — concetti e codice identici, cambia l'etichetta |
| 3 | **Tasso di default** del campione: ~8% — sbilanciamento realistico per un portafoglio retail, piu' severo del 22% del Taiwan |
| 4 | **Soglie di lettura del Gini** (debole < 40% ≤ accettabile ≤ 60% < buono): **prassi di validazione**, non valori imposti da una norma (ipotesi I1 del corso) |
| 5 | **Ambiente**: tutto gira su **CPU**; i blocchi di addestramento piu' lunghi impiegano qualche minuto e lo dichiarano in testa |
| 6 | **Riproducibilita'**: `SEED = 42` ovunque; stesso split 60/20/20 stratificato della mattina 1 |

## Glossario — acronimi e concetti usati in questa mattina

Le voci gia' definite nella mattina 1 (PD, default, Gini, AUC, PR-AUC, KS, Brier, calibrazione, reliability diagram, test binomiale e di Jeffreys, champion/challenger, train/validation/test, overfitting, tensore, autograd, loss, BCE, logit, epoca, batch, learning rate, Adam, MLP, ReLU, early stopping) **restano valide**: qui aggiungiamo solo i termini nuovi di oggi.

| Termine | Significato |
|---|---|
| **Imputazione** | riempire i valori mancanti con un valore plausibile (qui: la **mediana** calcolata sul train) |
| **Flag di mancanza** | colonna 0/1 che segnala «questo valore era assente»: l'assenza stessa puo' essere informativa |
| **Leakage** (fuga di informazione) | usare in addestramento informazioni che al momento della previsione non ci sarebbero (es. statistiche calcolate anche su validation/test); gonfia le metriche e si paga in produzione |
| **Cardinalita'** | numero di livelli distinti di una variabile categorica (es. `ORGANIZATION_TYPE`: 58) |
| **One-hot encoding** | una colonna 0/1 per ogni livello della categorica; esplode con l'alta cardinalita' |
| **Encoding ordinale** | livello → numero intero; compatto, ma inventa un ordine che non esiste (agli alberi non pesa: separano per soglie; a una rete o a un modello lineare si') |
| **Embedding** | rappresentazione **densa e appresa** di una categorica: ogni livello diventa un vettore di pochi numeri reali, aggiornato dall'addestramento come ogni altro peso |
| **Lookup table** (tabella di consultazione) | com'e' fatto un embedding dentro: una tabella «indice livello → vettore»; `nn.Embedding` e' esattamente questo |
| **Livello mai visto** (*unseen category*) | valore categorico assente nel train ma presente in produzione; riserviamo l'indice 0 per gestirlo senza errori |
| **`Dataset` / `DataLoader`** (PyTorch) | le classi che impacchettano i dati e li servono alla rete in batch mescolati; oggi scriviamo un `Dataset` nostro |
| **Dropout** | durante l'addestramento «spegne» a caso una quota di neuroni: la rete non puo' affidarsi a nessun neurone singolo → generalizza meglio |
| **Batch normalization** | normalizza le attivazioni dentro ogni batch: addestramento piu' stabile e veloce, lieve effetto regolarizzante |
| **Weight decay** | penalita' **L2** (somma dei quadrati dei pesi) aggiunta alla loss: scoraggia pesi grandi, modello piu' «liscio» |
| **Pazienza** (*patience*) | nell'early stopping: quante epoche aspettare senza miglioramenti prima di fermarsi |
| **Iperparametro** | scelta di progetto fissata prima dell'addestramento (larghezza, dropout, learning rate…); si sceglie sulla **validation**, mai sul test |
| **Esperimento controllato** | si cambia **una cosa alla volta** e si misura; e' il solo modo di sapere che cosa ha funzionato |
| **Pesi di classe / `pos_weight`** | moltiplicano la loss sugli esempi della classe rara: sbagliare un default «costa» di piu' |
| **Focal loss** | variante della BCE che riduce il peso degli esempi gia' ben classificati (fattore (1−p)^γ): concentra l'addestramento sui casi difficili |
| **Gamma (γ)** | l'intensita' della focal loss: γ = 0 riporta alla BCE; valori tipici 1–3 |
| **Platt scaling** | calibrazione: regressione logistica a una variabile sui **logit** del modello (adattata sulla validation) |
| **Regressione isotonica** | calibrazione: funzione a gradini **monotona** appresa da probabilita' stimata → frequenza osservata; piu' flessibile, chiede piu' dati |
| **Temperature scaling** | calibrazione minimalista per reti: si dividono i logit per una **temperatura T** (un solo parametro); T > 1 «raffredda» una rete troppo sicura di se' |
| **NLL / log-loss** | la verosimiglianza negativa: la quantita' che la calibrazione a temperatura minimizza |
| **Loan-to-value** | rapporto tra credito concesso e valore del bene (`AMT_CREDIT / AMT_GOODS_PRICE`): sopra 1 si finanzia piu' del valore |
| **Thin file** | cliente con poca storia creditizia: per lui anche i punteggi esterni (`EXT_SOURCE_*`) spesso mancano |
| **`EXT_SOURCE_1/2/3`** | punteggi di fonti esterne (tipo credit bureau), normalizzati 0–1: nel dataset reale sono i predittori piu' forti |
| **FT-Transformer, TabNet** | architetture di deep learning **dedicate al tabellare** (citate nella pillola 3): fuori perimetro oggi, il principio degli embedding e' lo stesso |

## 0. Verifica dell'ambiente

Importiamo le librerie della mattinata e fissiamo il seme. PyTorch verra' importato piu' avanti, quando entra in scena: tutta la prima parte (dati, preprocessing, champion) e' ancora mondo pandas/scikit-learn.

In [ ]:
# --- Import delle librerie della prima parte ------------------------------
# Ogni import e' commentato con il ruolo che la libreria avra' oggi.

import sys, platform
import numpy as np                      # calcolo numerico
import pandas as pd                     # tabelle (DataFrame)
import matplotlib.pyplot as plt         # grafici: missing, curve di apprendimento, reliability

from scipy import stats                 # test di calibrazione (binomiale, Jeffreys)
from scipy.optimize import minimize_scalar   # servira' per la temperatura (sezione 3)

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, average_precision_score,
                             brier_score_loss)
from sklearn.calibration import calibration_curve

import xgboost as xgb                   # il champion, come nella mattina 1

print("Python      :", sys.version.split()[0], "|", platform.system())
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost     :", xgb.__version__)

SEED = 42
np.random.seed(SEED)
print("\nAmbiente pronto. SEED =", SEED)

---
# Sezione 1 — Preprocessing per reti: mancanti, scaling, embedding

**Richiamo della pillola 1.** Gli alberi e le reti chiedono ai dati cose diverse, e chi viene da XGBoost deve ricalibrare tre riflessi:

1. **i valori mancanti**: XGBoost li gestisce nativamente (impara da che parte mandare i NaN a ogni split); una rete **no** — un NaN dentro una moltiplicazione produce NaN ovunque. Serve l'**imputazione**, piu' un **flag di mancanza** perche' l'assenza stessa e' spesso informativa;
2. **lo scaling**: irrilevante per gli alberi (decidono per soglie: «reddito > 30.000?»), **necessario** per le reti (apprendono per gradienti: variabili su scale diverse rendono la discesa instabile);
3. **le categoriche**: per gli alberi basta un encoding qualsiasi; per una rete il one-hot funziona finche' i livelli sono pochi, ma su `ORGANIZATION_TYPE` (58 livelli) esplode in colonne sparse e non impara nulla sui livelli rari. La risposta del deep learning sono gli **embedding**: ogni livello diventa un piccolo vettore denso **appreso durante l'addestramento** (Guo & Berkhahn, 2016).

Il dataset di oggi e' fatto apposta per toccare tutti e tre i punti.

## 1.1 Caricamento del dataset

Il loader cerca prima il campione reale fornito col corso, poi il segnaposto sintetico; in mancanza di entrambi si ferma con le istruzioni.

In [ ]:
# --- Caricamento dati -----------------------------------------------------
import os

PERCORSO_CAMPIONE   = "dati/home_credit_campione.csv"    # campione reale fornito col corso
PERCORSO_SEGNAPOSTO = "dati/home_credit_sintetico.csv"   # segnaposto per far girare il codice

if os.path.exists(PERCORSO_CAMPIONE):
    df = pd.read_csv(PERCORSO_CAMPIONE)
    print("Caricato il campione REALE:", PERCORSO_CAMPIONE)
elif os.path.exists(PERCORSO_SEGNAPOSTO):
    df = pd.read_csv(PERCORSO_SEGNAPOSTO)
    print("ATTENZIONE: caricato il SEGNAPOSTO SINTETICO:", PERCORSO_SEGNAPOSTO)
    print("I numeri che seguono servono solo a provare il codice, non a trarre conclusioni.")
else:
    raise FileNotFoundError(
        "Nessun dataset trovato. Posizionare il file del corso in " + PERCORSO_CAMPIONE)

print("\nDimensioni:", df.shape[0], "righe x", df.shape[1], "colonne")
print(f"Tasso di default (TARGET=1): {df['TARGET'].mean():.2%}")
df.head()

**Le colonne, in breve** (31 selezionate dalle 122 del file Kaggle originale):

- `SK_ID_CURR` — identificativo della domanda (non e' un predittore); `TARGET` — **il target**: 1 = difficolta' di pagamento;
- **importi**: `AMT_INCOME_TOTAL` (reddito annuo), `AMT_CREDIT` (credito richiesto), `AMT_ANNUITY` (rata annua), `AMT_GOODS_PRICE` (valore del bene finanziato);
- **anagrafica in giorni** (negativi = giorni *prima* della domanda): `DAYS_BIRTH` (eta'), `DAYS_EMPLOYED` (anzianita' lavorativa — con una sorpresa che vedremo), `DAYS_REGISTRATION`, `DAYS_ID_PUBLISH`, `DAYS_LAST_PHONE_CHANGE`;
- **punteggi esterni**: `EXT_SOURCE_1/2/3` — score di fonti esterne (tipo credit bureau) normalizzati 0–1: i predittori piu' forti, ma pieni di buchi;
- **famiglia e territorio**: `CNT_CHILDREN`, `CNT_FAM_MEMBERS`, `REGION_POPULATION_RELATIVE`, `REGION_RATING_CLIENT` (rating 1–3 della regione), `OBS/DEF_30_CNT_SOCIAL_CIRCLE` (quante persone della cerchia del cliente osservate / in ritardo), `AMT_REQ_CREDIT_BUREAU_YEAR` (interrogazioni al bureau nell'anno);
- **categoriche a bassa cardinalita'**: tipo contratto, genere, possesso auto/immobile, tipo reddito, istruzione, stato di famiglia, tipo abitazione;
- **categoriche ad alta cardinalita'** — le protagoniste di oggi: `OCCUPATION_TYPE` (18 livelli, 31% mancante) e `ORGANIZATION_TYPE` (58 livelli).

## 1.2 Prima ricognizione: mancanti e cardinalita'

Prima regola davanti a un dataset nuovo: guardare **dove mancano i dati** e **quanti livelli** hanno le categoriche. Sono le due cose che decidono il preprocessing.

In [ ]:
# --- Separiamo i nomi delle colonne per ruolo -------------------------------
COL_ID, COL_TARGET = "SK_ID_CURR", "TARGET"

COL_CATEGORICHE = ["NAME_CONTRACT_TYPE", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
                   "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
                   "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE"]
COL_NUMERICHE = [c for c in df.columns if c not in COL_CATEGORICHE + [COL_ID, COL_TARGET]]

print(f"{len(COL_NUMERICHE)} numeriche, {len(COL_CATEGORICHE)} categoriche\n")

# --- Valori mancanti, in percentuale (solo le colonne che ne hanno) ---------
pct_mancanti = (df.isna().mean() * 100).round(1)
pct_mancanti = pct_mancanti[pct_mancanti > 0].sort_values(ascending=False)
print("Percentuale di valori mancanti:")
print(pct_mancanti.to_string())

# --- Cardinalita' delle categoriche ------------------------------------------
print("\nCardinalita' (numero di livelli) delle categoriche:")
print(df[COL_CATEGORICHE].nunique().sort_values(ascending=False).to_string())

In [ ]:
# --- Il quadro dei mancanti, a colpo d'occhio --------------------------------
fig, ax = plt.subplots(figsize=(8, 3.6))
pct_mancanti.plot.barh(ax=ax)
ax.set_xlabel("% valori mancanti")
ax.set_title("Dove mancano i dati")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

# EXT_SOURCE_1 manca piu' di una volta su due, OCCUPATION_TYPE quasi una su tre:
# non sono eccezioni da nascondere, sono una caratteristica del fenomeno
# (molti richiedenti hanno poca storia creditizia: i 'thin file').

## 1.3 Un'anomalia da manuale: `DAYS_EMPLOYED` = 365243

Guardiamo la distribuzione dell'anzianita' lavorativa. C'e' un valore che torna identico decine di migliaia di volte: **365243 giorni**, cioe' *mille anni nel futuro*. E' il **codice tecnico** con cui la fonte marca chi non ha un impiego da misurare — in pratica i **pensionati** (incrociando con `NAME_INCOME_TYPE` si vede subito). Lasciarlo cosi' avvelenerebbe scaling e addestramento: mille anni pesano piu' di qualunque segnale vero.

Trattamento standard, in tre mosse: **(1)** un flag esplicito `FLAG_PENSIONATO`; **(2)** il codice tecnico diventa `NaN` (verra' imputato come gli altri mancanti); **(3)** gia' che ci siamo, trasformiamo i giorni in **anni leggibili** e creiamo due rapporti dal significato creditizio immediato: la **rata su reddito** e il **loan-to-value**.

In [ ]:
# --- L'anomalia, vista -------------------------------------------------------
n_anomali = (df["DAYS_EMPLOYED"] == 365243).sum()
print(f"Righe con DAYS_EMPLOYED == 365243: {n_anomali:,} ({n_anomali/len(df):.1%})")
print("\nChi sono? NAME_INCOME_TYPE delle righe anomale:")
print(df.loc[df["DAYS_EMPLOYED"] == 365243, "NAME_INCOME_TYPE"].value_counts().head().to_string())

# --- Trattamento + variabili derivate ---------------------------------------
df["FLAG_PENSIONATO"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df.loc[df["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan

df["ETA_ANNI"]        = -df["DAYS_BIRTH"] / 365.25          # giorni negativi -> anni positivi
df["ANZIANITA_ANNI"]  = -df["DAYS_EMPLOYED"] / 365.25
df["RATA_SU_REDDITO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df["LOAN_TO_VALUE"]   = df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"]

# le originali trasformate escono di scena, le derivate entrano nell'elenco
df = df.drop(columns=["DAYS_BIRTH", "DAYS_EMPLOYED"])
COL_NUMERICHE = [c for c in COL_NUMERICHE if c not in ("DAYS_BIRTH", "DAYS_EMPLOYED")] +                 ["FLAG_PENSIONATO", "ETA_ANNI", "ANZIANITA_ANNI", "RATA_SU_REDDITO", "LOAN_TO_VALUE"]

print(f"\nDopo il trattamento: {len(COL_NUMERICHE)} numeriche.")
df[["FLAG_PENSIONATO", "ETA_ANNI", "ANZIANITA_ANNI", "RATA_SU_REDDITO", "LOAN_TO_VALUE"]].describe().round(3)

## 1.4 Split 60/20/20 — prima di qualunque statistica

Stesso schema igienico della mattina 1: **train** (60%) per addestrare, **validation** (20%) per le scelte, **test** (20%) per il giudizio finale, stratificando sul target.

Un punto merita l'evidenza: lo split va fatto **prima** di calcolare mediane per l'imputazione, medie per lo scaling, mappe dei livelli categorici. Ogni statistica calcolata anche su validation o test e' **leakage**: informazione che in produzione non avremmo, e che gonfia silenziosamente le metriche.

In [ ]:
# --- Split stratificato 60/20/20 ---------------------------------------------
X = df.drop(columns=[COL_TARGET, COL_ID])
y = df[COL_TARGET]

X_train, X_resto, y_train, y_resto = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_resto, y_resto, test_size=0.50, stratify=y_resto, random_state=SEED)

for nome, parte in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{nome:11s}: {len(parte):6,} domande | tasso default {parte.mean():.2%}")

## 1.5 La cassetta degli attrezzi della mattina 1

Ricopiamo qui — identiche — le tre funzioni costruite ieri, cosi' il notebook e' autosufficiente: `valuta_modello()` (tutte le metriche), `semaforo_gini()` (la lettura di prassi) e `tabella_calibrazione()` (fasce di rating + test binomiale e di Jeffreys).

In [ ]:
# --- Cassetta degli attrezzi (dalla mattina 1) -------------------------------
def valuta_modello(y_vero, proba, soglia=0.5):
    """Restituisce un dizionario con tutte le metriche del corso."""
    classe = (np.asarray(proba) >= soglia).astype(int)
    out = {
        "accuracy":  accuracy_score(y_vero, classe),
        "precision": precision_score(y_vero, classe, zero_division=0),
        "recall":    recall_score(y_vero, classe),
        "F1":        f1_score(y_vero, classe),
    }
    auc = roc_auc_score(y_vero, proba)
    out["AUC"]    = auc
    out["Gini"]   = 2 * auc - 1
    out["PR-AUC"] = average_precision_score(y_vero, proba)
    fpr, tpr, _ = roc_curve(y_vero, proba)
    out["KS"]    = float(np.max(tpr - fpr))
    out["Brier"] = brier_score_loss(y_vero, proba)
    return out

def semaforo_gini(gini):
    """Lettura di prassi del Gini (ipotesi I1 del corso: prassi, non norma)."""
    if gini < 0.40:  return "DEBOLE (< 40%)"
    if gini <= 0.60: return "ACCETTABILE (40-60%)"
    return "BUONO (> 60%)"

def tabella_calibrazione(y_vero, proba, n_fasce=8):
    """Tabella per fascia di rating: test binomiale e test di Jeffreys (istruzioni BCE)."""
    tab = pd.DataFrame({"proba": np.asarray(proba), "y": np.asarray(y_vero)})
    tab["fascia"] = pd.qcut(tab["proba"], q=n_fasce, labels=False, duplicates="drop") + 1
    righe = []
    for f, g in tab.groupby("fascia"):
        n, d = len(g), int(g["y"].sum())
        pd_stimata = g["proba"].mean()
        p_binom = stats.binomtest(d, n, pd_stimata).pvalue
        p_jeff  = stats.beta.cdf(pd_stimata, d + 0.5, n - d + 0.5)
        righe.append({"fascia": int(f), "n": n, "default": d,
                      "PD stimata": pd_stimata, "tasso osservato": g["y"].mean(),
                      "p binomiale": p_binom, "p Jeffreys": p_jeff})
    out = pd.DataFrame(righe).set_index("fascia")
    out["esito"] = np.where(out["p Jeffreys"] < 0.05, "SOTTOSTIMA?", "ok")
    return out

print("Cassetta pronta: valuta_modello(), semaforo_gini(), tabella_calibrazione().")

## 1.6 Il champion sul nuovo terreno (e la logistica di controllo)

Nuovo dataset ⇒ il confronto riparte **a parita' di dati**: riaddestriamo XGBoost qui (stessi iperparametri della mattina 1) e teniamo la regressione logistica come riferimento lineare. Notare la differenza di trattamento richiesta dai due mondi:

- **XGBoost**: nessuna imputazione (i NaN li gestisce da solo) e **encoding ordinale** delle categoriche — l'ordine fittizio introdotto non lo disturba, perche' un albero separa i livelli per soglie successive;
- **logistica**: da modello lineare, vuole il percorso completo — imputazione, **one-hot** e scaling — che impacchettiamo in una `Pipeline` con `ColumnTransformer` (cosi' ogni statistica e' adattata sul solo train, senza leakage).

In [ ]:
# --- Champion: XGBoost con encoding ordinale ---------------------------------
# OrdinalEncoder: livello -> intero; i livelli mai visti diventano -1.
# Nota: astype(str) trasforma le categoriche mancanti nel livello "nan", che
# l'encoder codifica come un livello qualsiasi (per un albero va benissimo);
# i NaN NUMERICI invece restano NaN: quelli XGBoost li gestisce nativamente.
encoder_ord = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder_ord.fit(X_train[COL_CATEGORICHE].astype(str))

def per_xgboost(X_df):
    """Matrice per XGBoost: categoriche in codici ordinali, NaN numerici lasciati ai NaN."""
    Xm = X_df.copy()
    Xm[COL_CATEGORICHE] = encoder_ord.transform(Xm[COL_CATEGORICHE].astype(str))
    return Xm

champion = xgb.XGBClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="auc", random_state=SEED)
champion.fit(per_xgboost(X_train), y_train)
proba_xgb_val  = champion.predict_proba(per_xgboost(X_val))[:, 1]
proba_xgb_test = champion.predict_proba(per_xgboost(X_test))[:, 1]
print("XGBoost (champion) addestrato.")

# --- Riferimento lineare: logistica in pipeline completa ---------------------
prep_lineare = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), COL_NUMERICHE),
    ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="(mancante)"),
                          OneHotEncoder(handle_unknown="ignore")), COL_CATEGORICHE),
])
logistica = make_pipeline(prep_lineare, LogisticRegression(max_iter=3000, random_state=SEED))
logistica.fit(X_train, y_train)
proba_log_val  = logistica.predict_proba(X_val)[:, 1]
proba_log_test = logistica.predict_proba(X_test)[:, 1]
print("Logistica addestrata.")

# --- Prime misure (sulla VALIDATION: il test resta sigillato fino alla fine) --
prime = pd.DataFrame({
    "Logistica": valuta_modello(y_val, proba_log_val),
    "XGBoost":   valuta_modello(y_val, proba_xgb_val),
}).T.round(4)
print("\nMisure su validation:")
print(prime[["Gini", "PR-AUC", "KS", "Brier"]].to_string())
for nome, p in [("Logistica", proba_log_val), ("XGBoost  ", proba_xgb_val)]:
    g = 2 * roc_auc_score(y_val, p) - 1
    print(f"Gini {nome}: {g:.1%}  ->  {semaforo_gini(g)}")

**Nota di lettura.** Con un tasso di default all'8% le metriche *a soglia* 0,5 (precision, recall, F1) sono quasi mute — pochissime probabilita' stimate superano 0,5. Nessun dramma: come fissato nella mattina 1, per una PD guardiamo **Gini, PR-AUC, KS** (ordinamento) e **Brier** (probabilita'). L'asticella del champion e' fissata.

## 1.7 Preprocessing per la rete: imputazione, flag, scaling, indici

Ora prepariamo gli **stessi dati** per il challenger. Il percorso, tutto adattato **sul solo train**:

1. **numeriche**: mediana del train sui buchi + **flag di mancanza** per le colonne con assenze rilevanti; poi `StandardScaler`;
2. **categoriche**: i NaN diventano il livello esplicito `"(mancante)"` (su `OCCUPATION_TYPE` e' quasi un terzo delle righe: un livello a pieno titolo); poi ogni livello diventa un **indice intero** tramite mappe costruite sul train, con l'**indice 0 riservato** ai livelli mai visti — cosi' un valore nuovo in produzione non manda in errore la rete.

In [ ]:
# --- Numeriche: imputazione con mediana del train + flag + scaling ------------
mediane_train = X_train[COL_NUMERICHE].median()
COL_CON_BUCHI = [c for c in COL_NUMERICHE if X_train[c].isna().mean() > 0.01]
print("Colonne con flag di mancanza:", COL_CON_BUCHI)

def prepara_numeriche(X_df):
    """Imputa con le mediane del train e aggiunge i flag di mancanza (0/1)."""
    num = X_df[COL_NUMERICHE].copy()
    for c in COL_CON_BUCHI:
        num[c + "_MANCANTE"] = X_df[c].isna().astype(int)
    return num.fillna(mediane_train)

num_train = prepara_numeriche(X_train)
scaler = StandardScaler().fit(num_train)              # adattato SOLO sul train

def numeriche_scalate(X_df):
    return scaler.transform(prepara_numeriche(X_df)).astype(np.float32)

print(f"Matrice numerica per la rete: {numeriche_scalate(X_train).shape[1]} colonne "
      f"({len(COL_NUMERICHE)} variabili + {len(COL_CON_BUCHI)} flag)")

In [ ]:
# --- Categoriche: mappe livello -> indice (0 = mai visto) ---------------------
def costruisci_mappa(serie):
    """Mappa {livello: indice} dai livelli del TRAIN; l'indice 0 resta libero
    per i livelli mai visti (o si usa per '(mancante)' se mai visto anch'esso)."""
    livelli = sorted(serie.fillna("(mancante)").astype(str).unique())
    return {liv: i + 1 for i, liv in enumerate(livelli)}      # indici da 1; 0 = sconosciuto

mappe = {c: costruisci_mappa(X_train[c]) for c in COL_CATEGORICHE}
cardinalita = {c: len(m) + 1 for c, m in mappe.items()}        # +1 per l'indice 0

def categoriche_in_indici(X_df):
    """Applica le mappe del train: livello -> indice; mai visto -> 0."""
    out = np.zeros((len(X_df), len(COL_CATEGORICHE)), dtype=np.int64)
    for j, c in enumerate(COL_CATEGORICHE):
        valori = X_df[c].fillna("(mancante)").astype(str)
        out[:, j] = valori.map(mappe[c]).fillna(0).astype(np.int64)
    return out

print("Cardinalita' (incluso lo 0 riservato ai mai visti):")
for c in COL_CATEGORICHE:
    print(f"  {c:22s}: {cardinalita[c]:3d}")

## 1.8 Gli embedding: la categorica diventa geometria

Il **one-hot** di `ORGANIZATION_TYPE` produrrebbe 58 colonne quasi tutte a zero, e per la rete «School» e «University» sarebbero distanti quanto «School» e «Construction»: nessuna nozione di somiglianza. L'**embedding** ribalta la prospettiva: a ogni livello associamo un vettore di pochi numeri reali (qui da 2 a 16 dimensioni, secondo la regola pratica `min(16, (cardinalita'+1)//2)`), e quei numeri sono **parametri appresi** — l'addestramento avvicina da solo i livelli che si comportano in modo simile rispetto al default.

Tecnicamente `nn.Embedding(cardinalita', dimensione)` e' una **lookup table**: riga *i* = vettore del livello *i*. La useremo cosi': una tabella per ciascuna delle 10 categoriche, i 10 vettori concatenati alle numeriche scalate, e il tutto in pasto al tronco MLP.

Da qui entra in scena PyTorch: costruiamo il `Dataset` su misura (restituisce la tripla *numeriche, indici categorici, target*) e il `DataLoader` che serve i batch.

In [ ]:
# --- PyTorch: Dataset su misura e DataLoader ----------------------------------
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(SEED)
print("PyTorch", torch.__version__, "| GPU disponibile:", torch.cuda.is_available(),
      "(oggi lavoriamo volutamente su CPU)")

class DatasetCredito(Dataset):
    """Impacchetta (numeriche scalate, indici categorici, target) come tensori."""

    def __init__(self, X_df, y_serie):
        self.x_num = torch.tensor(numeriche_scalate(X_df))                  # float32
        self.x_cat = torch.tensor(categoriche_in_indici(X_df))              # int64
        self.y     = torch.tensor(y_serie.to_numpy(), dtype=torch.float32).reshape(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.x_num[i], self.x_cat[i], self.y[i]

ds_train = DatasetCredito(X_train, y_train)
ds_val   = DatasetCredito(X_val,   y_val)
ds_test  = DatasetCredito(X_test,  y_test)

loader_train = DataLoader(ds_train, batch_size=512, shuffle=True)

print("Dataset pronti:", len(ds_train), "train |", len(ds_val), "val |", len(ds_test), "test")
print("Un esempio:", tuple(ds_train[0][0].shape), "numeriche +",
      tuple(ds_train[0][1].shape), "indici categorici")

In [ ]:
# --- La rete con embedding -----------------------------------------------------
def dimensione_embedding(card):
    """Regola pratica: meta' della cardinalita', tra 2 e 16."""
    return int(min(16, max(2, (card + 1) // 2)))

class RetePDEmbedding(nn.Module):
    """MLP con embedding per le categoriche.

    Parametri gia' predisposti per la sezione 2:
      hidden     : larghezze degli strati nascosti
      p_dropout  : quota di neuroni spenti dal dropout (0 = spento)
      batchnorm  : se True, BatchNorm1d dopo ogni strato lineare
    """

    def __init__(self, cardinalita, n_numeriche, hidden=(128, 64),
                 p_dropout=0.0, batchnorm=False):
        super().__init__()
        # una lookup table per ciascuna categorica
        self.embedding = nn.ModuleList([
            nn.Embedding(card, dimensione_embedding(card)) for card in cardinalita])
        n_da_embedding = sum(dimensione_embedding(c) for c in cardinalita)

        # tronco MLP costruito strato per strato
        strati, n_in = [], n_numeriche + n_da_embedding
        for n_out in hidden:
            strati.append(nn.Linear(n_in, n_out))
            if batchnorm:
                strati.append(nn.BatchNorm1d(n_out))
            strati.append(nn.ReLU())
            if p_dropout > 0:
                strati.append(nn.Dropout(p_dropout))
            n_in = n_out
        strati.append(nn.Linear(n_in, 1))               # uscita: UN logit
        self.tronco = nn.Sequential(*strati)

    def forward(self, x_num, x_cat):
        # ogni colonna di indici attraversa la SUA lookup table
        vettori = [emb(x_cat[:, j]) for j, emb in enumerate(self.embedding)]
        x = torch.cat([x_num] + vettori, dim=1)         # numeriche + embedding, affiancati
        return self.tronco(x)

carte = [cardinalita[c] for c in COL_CATEGORICHE]
rete_base = RetePDEmbedding(carte, n_numeriche=ds_train.x_num.shape[1])
print(rete_base)
print(f"\nParametri da apprendere: {sum(p.numel() for p in rete_base.parameters()):,}")
print("Dimensioni embedding:", {c: dimensione_embedding(cardinalita[c]) for c in COL_CATEGORICHE})

## 1.9 Il training loop della mattina 1, impacchettato

Il ciclo e' lo **stesso identico** scritto per esteso ieri (forward → loss → zero_grad → backward → step, piu' early stopping sulla validation). Oggi lo impacchettiamo in una funzione `addestra()` con due aggiunte da laboratorio:

- registriamo anche la **loss di validation** (ci servira' per *vedere* l'overfitting);
- la **pazienza**: se l'AUC di validation non migliora per `pazienza` epoche consecutive, ci fermiamo davvero (ieri ci limitavamo a ricordare i pesi migliori).

In [ ]:
# --- Funzione di addestramento riusabile ---------------------------------------
import copy, time

def addestra(rete, loader, ds_validation, epoche=30, lr=1e-3,
             weight_decay=0.0, loss_fn=None, pazienza=5, verboso=True):
    """Addestra la rete con early stopping sull'AUC di validation.

    Restituisce (storia, epoca_migliore); la rete esce con i PESI MIGLIORI caricati.
    weight_decay : penalita' L2 passata all'ottimizzatore Adam
    loss_fn      : funzione di loss (default BCEWithLogitsLoss)
    pazienza     : epoche senza miglioramenti prima dello stop
    """
    if loss_fn is None:
        loss_fn = nn.BCEWithLogitsLoss()
    ottimizza = torch.optim.Adam(rete.parameters(), lr=lr, weight_decay=weight_decay)
    storia = {"loss_train": [], "loss_val": [], "auc_val": []}
    migliore = {"auc": -1.0, "pesi": None, "epoca": -1}
    x_num_val, x_cat_val, y_val_t = ds_validation.x_num, ds_validation.x_cat, ds_validation.y

    via = time.time()
    for epoca in range(1, epoche + 1):
        rete.train()
        loss_epoca = 0.0
        for x_num, x_cat, y_b in loader:
            logit = rete(x_num, x_cat)                 # 1) forward
            loss  = loss_fn(logit, y_b)                # 2) loss
            ottimizza.zero_grad()                      # 3) azzera
            loss.backward()                            # 4) gradienti
            ottimizza.step()                           # 5) aggiorna
            loss_epoca += loss.item() * len(y_b)
        loss_epoca /= len(loader.dataset)

        rete.eval()
        with torch.no_grad():
            logit_val = rete(x_num_val, x_cat_val)
            loss_val  = loss_fn(logit_val, y_val_t).item()
            proba_val = torch.sigmoid(logit_val).numpy().ravel()
        auc_val = roc_auc_score(y_val_t.numpy().ravel(), proba_val)

        storia["loss_train"].append(loss_epoca)
        storia["loss_val"].append(loss_val)
        storia["auc_val"].append(auc_val)

        if auc_val > migliore["auc"]:
            migliore = {"auc": auc_val, "pesi": copy.deepcopy(rete.state_dict()), "epoca": epoca}
        elif epoca - migliore["epoca"] >= pazienza:
            if verboso:
                print(f"  early stopping all'epoca {epoca} (migliore: {migliore['epoca']})")
            break
        if verboso and (epoca % 5 == 0 or epoca == 1):
            print(f"  epoca {epoca:2d} | loss train {loss_epoca:.4f} | "
                  f"loss val {loss_val:.4f} | AUC val {auc_val:.4f}")

    rete.load_state_dict(migliore["pesi"])
    if verboso:
        print(f"  pesi migliori: epoca {migliore['epoca']} "
              f"(AUC val {migliore['auc']:.4f}) | {time.time()-via:.0f}s")
    return storia, migliore["epoca"]

def proba_rete(rete, ds):
    """Probabilita' della rete su un intero DatasetCredito (senza gradienti)."""
    rete.eval()
    with torch.no_grad():
        return torch.sigmoid(rete(ds.x_num, ds.x_cat)).numpy().ravel()

print("Funzioni pronte: addestra() e proba_rete().")

In [ ]:
# --- Primo giro: la rete con embedding, ancora senza regolarizzazione ----------
# (~1-2 minuti su CPU)
torch.manual_seed(SEED)
rete_base = RetePDEmbedding(carte, n_numeriche=ds_train.x_num.shape[1])
storia_base, _ = addestra(rete_base, loader_train, ds_val, epoche=30)

gini_base = 2 * roc_auc_score(y_val, proba_rete(rete_base, ds_val)) - 1
gini_xgb  = 2 * roc_auc_score(y_val, proba_xgb_val) - 1
print(f"\nGini validation | rete con embedding: {gini_base:.1%} | champion XGBoost: {gini_xgb:.1%}")

**Dove siamo.** Gia' con i soli embedding il challenger non e' piu' quello disarmato di ieri: il grosso del distacco dal champion si e' chiuso (o quasi — i numeri esatti dipendono dal campione). La sezione 2 serve a spremere il resto: controllare l'overfitting e affrontare lo sbilanciamento.

---
# Sezione 2 — Regolarizzazione e sbilanciamento, con metodo

**Richiamo della pillola 2.** Una rete ha molti piu' parametri di quanti ne servano per *memorizzare* il rumore del train: prima o poi lo fara' — e' l'**overfitting**. Gli antidoti si chiamano **dropout**, **batch normalization**, **weight decay**, **early stopping**. Il punto di metodo, caro a qualunque validatore: si prova **una tecnica alla volta**, a parita' di tutto il resto, e si misura. Le conclusioni «a sensazione» non si difendono; una tabella di esperimenti controllati si'.

## 2.1 L'overfitting, visto in faccia

Per vederlo chiaramente lo *provochiamo*: rete piu' larga (256→128), niente regolarizzazione, niente early stopping (pazienza altissima), 40 epoche. Da manuale: la loss di train scende sempre, quella di validation tocca il minimo e **risale** — da li' in poi la rete sta imparando il rumore.

*(Questa cella e' la piu' lunga della mattinata: ~2-3 minuti su CPU.)*

In [ ]:
# --- Overfitting provocato ------------------------------------------------------
torch.manual_seed(SEED)
rete_larga = RetePDEmbedding(carte, n_numeriche=ds_train.x_num.shape[1], hidden=(256, 128))
storia_of, _ = addestra(rete_larga, loader_train, ds_val,
                        epoche=40, pazienza=40, verboso=False)   # pazienza 40 = mai fermarsi

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
epoche_ = range(1, len(storia_of["loss_train"]) + 1)
ax1.plot(epoche_, storia_of["loss_train"], label="loss train")
ax1.plot(epoche_, storia_of["loss_val"],  label="loss validation")
minimo = int(np.argmin(storia_of["loss_val"])) + 1
ax1.axvline(minimo, color="red", ls="--", lw=1, label=f"minimo val: epoca {minimo}")
ax1.set_xlabel("epoca"); ax1.set_ylabel("BCE"); ax1.set_title("Le due loss"); ax1.legend()

ax2.plot(epoche_, storia_of["auc_val"], color="tab:green")
ax2.set_xlabel("epoca"); ax2.set_ylabel("AUC validation"); ax2.set_title("AUC su validation")
plt.tight_layout(); plt.show()

# La forbice tra le due curve DOPO il minimo e' l'overfitting: la rete migliora
# sul train e peggiora sui dati nuovi. Tutto cio' che segue serve a chiuderla.

## 2.2 Esperimenti controllati: una tecnica alla volta

Configurazione per configurazione, **cambia una sola cosa**; stessa architettura (128→64), stesso seed, stesso loader, stessa pazienza. Misuriamo il **Gini di validation** coi pesi migliori di ciascuna.

*(Cinque addestramenti in fila: ~4-6 minuti su CPU. Mandiamo in esecuzione e commentiamo intanto la tabella attesa.)*

In [ ]:
# --- Griglia di esperimenti: una tecnica alla volta -------------------------------
esperimenti = {
    "base (nessuna regolarizzazione)": dict(),
    "+ dropout 0.3":                   dict(p_dropout=0.3),
    "+ batch normalization":           dict(batchnorm=True),
    "+ weight decay 1e-4":             dict(weight_decay=1e-4),
    "dropout + batchnorm + wd":        dict(p_dropout=0.3, batchnorm=True, weight_decay=1e-4),
}

risultati_reg, storie_reg = {}, {}
for nome, cfg in esperimenti.items():
    wd = cfg.pop("weight_decay", 0.0)         # il weight decay va all'ottimizzatore...
    torch.manual_seed(SEED)                   # ...il resto alla rete
    rete_e = RetePDEmbedding(carte, n_numeriche=ds_train.x_num.shape[1],
                             hidden=(128, 64), **cfg)
    print(nome)
    storia_e, ep_best = addestra(rete_e, loader_train, ds_val,
                                 epoche=30, weight_decay=wd, verboso=False)
    p_val = proba_rete(rete_e, ds_val)
    risultati_reg[nome] = {
        "Gini val":   2 * roc_auc_score(y_val, p_val) - 1,
        "Brier val":  brier_score_loss(y_val, p_val),
        "epoca best": ep_best,
        "epoche eseguite": len(storia_e["auc_val"]),
    }
    storie_reg[nome] = storia_e

tab_reg = pd.DataFrame(risultati_reg).T
tab_reg["Gini val"]  = tab_reg["Gini val"].astype(float).round(4)
tab_reg["Brier val"] = tab_reg["Brier val"].astype(float).round(5)
print("\n" + tab_reg.to_string())

In [ ]:
# --- Le curve di validation, sovrapposte -------------------------------------------
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for nome, st in storie_reg.items():
    ax.plot(range(1, len(st["auc_val"]) + 1), st["auc_val"], label=nome)
ax.set_xlabel("epoca"); ax.set_ylabel("AUC validation")
ax.set_title("Esperimenti controllati - effetto di ogni tecnica")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# Lettura tipica (i numeri esatti dipendono dal campione):
# - dropout e weight decay: curve piu' PIATTE dopo il picco -> meno overfitting,
#   early stopping meno critico;
# - batch normalization: salita piu' RAPIDA nelle prime epoche;
# - la combinazione vince di poco o pareggia: su dati tabellari medi la
#   regolarizzazione COMPRA STABILITA' piu' che punti di Gini.

## 2.3 Lo sbilanciamento: pesi di classe e focal loss

Con l'8% di default, nei batch da 512 la rete vede ~41 default e ~471 buoni: la BCE e' dominata dalla classe facile. Due rimedi, dalla stessa famiglia (riscrivere la loss):

- **`pos_weight`**: la BCE pesa gli errori sui default per un fattore (qui `n_buoni/n_default` ≈ 11): e' il gemello del `class_weight` di scikit-learn. Conseguenza da conoscere: le probabilita' escono **gonfiate** (il modello «vede» un mondo riequilibrato) — la calibrazione della sezione 3 esiste anche per questo;
- **focal loss** (Lin et al., 2017): nata per l'object detection, riduce il peso degli esempi **gia' ben classificati** di un fattore (1−p)^γ. Non esiste pronta in PyTorch per il caso binario: la scriviamo noi — ed e' un ottimo esercizio di lettura di una loss custom.

In [ ]:
# --- Focal loss binaria, scritta a mano --------------------------------------------
class FocalLoss(nn.Module):
    """Focal loss per classificazione binaria su logit (Lin et al., 2017).

    FL(p_t) = -alfa_t * (1 - p_t)^gamma * log(p_t)
    dove p_t e' la probabilita' assegnata alla CLASSE VERA:
      - esempio facile  (p_t -> 1): (1-p_t)^gamma ~ 0  -> contributo quasi nullo
      - esempio difficile (p_t -> 0): fattore ~ 1      -> contributo pieno
    gamma = 0 riporta esattamente alla BCE pesata con alfa.
    """

    def __init__(self, gamma=2.0, alfa=0.25):
        super().__init__()
        self.gamma, self.alfa = gamma, alfa

    def forward(self, logit, y):
        # BCE per esempio (senza riduzione): -log(p_t)
        bce = nn.functional.binary_cross_entropy_with_logits(logit, y, reduction="none")
        p_t = torch.exp(-bce)                            # dalla BCE risaliamo a p_t
        alfa_t = self.alfa * y + (1 - self.alfa) * (1 - y)   # peso di classe
        return (alfa_t * (1 - p_t) ** self.gamma * bce).mean()

print("FocalLoss pronta (gamma=2, alfa=0.25 come nel paper).")

In [ ]:
# --- Confronto: BCE vs pos_weight vs focal (regolarizzazione fissata) ---------------
# (~3-4 minuti su CPU: tre addestramenti)
rapporto = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"pos_weight = n_buoni / n_default = {rapporto:.1f}\n")

varianti_loss = {
    "BCE semplice":            nn.BCEWithLogitsLoss(),
    f"pos_weight {rapporto:.0f}": nn.BCEWithLogitsLoss(pos_weight=torch.tensor([rapporto])),
    "focal (gamma=2)":         FocalLoss(gamma=2.0, alfa=0.25),
}

reti_sb, risultati_sb = {}, {}
for nome, lf in varianti_loss.items():
    torch.manual_seed(SEED)
    rete_s = RetePDEmbedding(carte, n_numeriche=ds_train.x_num.shape[1],
                             hidden=(128, 64), p_dropout=0.3)
    print(nome)
    addestra(rete_s, loader_train, ds_val, epoche=30,
             weight_decay=1e-4, loss_fn=lf, verboso=False)
    p_val = proba_rete(rete_s, ds_val)
    m = valuta_modello(y_val, p_val)
    risultati_sb[nome] = {k: m[k] for k in ("Gini", "PR-AUC", "KS", "recall", "Brier")}
    risultati_sb[nome]["PD media stimata"] = float(p_val.mean())
    reti_sb[nome] = rete_s

tab_sb = pd.DataFrame(risultati_sb).T.round(4)
print("\n" + tab_sb.to_string())
print(f"\nTasso di default reale della validation: {y_val.mean():.4f}")

**Lettura della tabella — il punto piu' importante della sezione.**

- Su **Gini / PR-AUC / KS** (metriche di *ordinamento*) le tre loss si equivalgono quasi sempre: riscalare la loss non insegna alla rete a ordinare meglio, sposta solo la «soglia interna»;
- il **recall a soglia 0,5** invece esplode con `pos_weight` e focal: utile per un *sistema di allerta* che deve classificare, non per una PD;
- il prezzo e' nella colonna **PD media stimata**: con `pos_weight` ≈ 11 la rete stima in media una PD vicina al 40–50% contro un tasso reale dell'8% — **Brier pessimo, probabilita' inservibili** per pricing e accantonamenti.

Morale operativa per un team PD: lo sbilanciamento all'8% **non e' un'emergenza** (lo era il low-default della pillola 3); se si riscala la loss, poi **si deve ricalibrare**. Che e' esattamente la sezione 3.

---
# Sezione 3 — Calibrare l'output e il verdetto finale

**Richiamo della pillola 3.** Una PD alimenta pricing, accantonamenti e capitale: le **probabilita' devono essere vere**, non solo ben ordinate. Le reti escono spesso *overconfident* anche senza pesi (Guo et al., 2017); con `pos_weight` escono gonfiate per costruzione. La buona notizia: la calibrazione e' un **post-processing** — si adatta una funzione sulle probabilita' di **validation** e la si applica al test, senza toccare la rete. Tre strumenti, dal piu' rigido al piu' flessibile:

| Metodo | Che cosa adatta | Quando preferirlo |
|---|---|---|
| **Temperature scaling** | un solo parametro T che divide i logit | reti overconfident «in blocco»; preserva l'ordinamento per costruzione |
| **Platt scaling** | logistica a 2 parametri sui logit | distorsione monotona semplice; pochi dati |
| **Isotonica** | funzione a gradini monotona | distorsioni di forma qualsiasi; chiede piu' dati (qui ne abbiamo) |

Scegliamo come challenger la rete **regolarizzata con `pos_weight`** — il caso piu' istruttivo: ordina bene e stima malissimo.

In [ ]:
# --- La rete da calibrare: logit e probabilita' su val e test -----------------------
rete_finale = reti_sb[[k for k in reti_sb if k.startswith("pos_weight")][0]]

def logit_rete(rete, ds):
    rete.eval()
    with torch.no_grad():
        return rete(ds.x_num, ds.x_cat).numpy().ravel()

logit_val_r,  logit_test_r  = logit_rete(rete_finale, ds_val),  logit_rete(rete_finale, ds_test)
proba_val_r,  proba_test_r  = 1/(1+np.exp(-logit_val_r)), 1/(1+np.exp(-logit_test_r))

print(f"PD media stimata (val): {proba_val_r.mean():.1%} | tasso reale: {y_val.mean():.1%}")
print(f"Brier (val) prima della calibrazione: {brier_score_loss(y_val, proba_val_r):.4f}")

In [ ]:
# --- I tre calibratori, adattati sulla SOLA validation -------------------------------
# 1) Temperature scaling: cerchiamo la T>0 che minimizza la log-loss di validation
def nll_con_T(T):
    p = 1 / (1 + np.exp(-logit_val_r / T))
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y_val * np.log(p) + (1 - y_val) * np.log(1 - p))

T_best = minimize_scalar(nll_con_T, bounds=(0.5, 50.0), method="bounded").x
print(f"Temperatura ottima: T = {T_best:.2f}  (T>1 = la rete era troppo 'calda')")

def con_temperatura(logit):
    return 1 / (1 + np.exp(-logit / T_best))

# NOTA: con pos_weight i logit sono TRASLATI, non solo troppo ampi; la sola T
# non puo' correggere una traslazione -> ci aspettiamo che Platt faccia meglio.

# 2) Platt scaling: logistica a 1 variabile sui logit (pendenza + intercetta)
platt = LogisticRegression(max_iter=1000)
platt.fit(logit_val_r.reshape(-1, 1), y_val)
def con_platt(logit):
    return platt.predict_proba(logit.reshape(-1, 1))[:, 1]

# 3) Isotonica: funzione monotona a gradini da proba stimata a frequenza osservata
isotonica = IsotonicRegression(out_of_bounds="clip")
isotonica.fit(proba_val_r, y_val)
def con_isotonica(proba):
    return isotonica.predict(proba)

print("Calibratori adattati su validation. Ora si applicano al TEST, mai riadattati.")

In [ ]:
# --- Prima/dopo sul test: Brier e reliability diagram --------------------------------
candidati = {
    "non calibrata":        proba_test_r,
    f"temperatura (T={T_best:.1f})": con_temperatura(logit_test_r),
    "Platt":                con_platt(logit_test_r),
    "isotonica":            con_isotonica(proba_test_r),
}

print(f"{'metodo':<22} {'Brier test':>10} {'PD media':>9} {'Gini':>7}")
for nome, p in candidati.items():
    g = 2 * roc_auc_score(y_test, p) - 1
    print(f"{nome:<22} {brier_score_loss(y_test, p):>10.4f} {p.mean():>8.1%} {g:>7.1%}")
print(f"{'(tasso reale test)':<22} {'':>10} {y_test.mean():>8.1%}")

fig, ax = plt.subplots(figsize=(6, 4.8))
for nome, p in candidati.items():
    frac_oss, pd_media = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    ax.plot(pd_media, frac_oss, "o-", ms=3.5, label=nome)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="calibrazione perfetta")
ax.set_xscale("log"); ax.set_yscale("log")        # scala log: le PD vivono vicino allo 0
ax.set_xlabel("PD media stimata (per fascia)"); ax.set_ylabel("tasso osservato")
ax.set_title("Reliability diagram sul test - prima e dopo")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# Il Gini NON cambia (o quasi): le calibrazioni monotone non toccano l'ordinamento.
# Cambia il Brier, cioe' la VERITA' delle probabilita'. Discriminazione e
# calibrazione sono proprieta' separate: servono entrambe, si misurano separate.

## 3.1 Il verdetto champion/challenger

La tabella che un validatore si aspetta: champion e challenger **sugli stessi dati di test, con le stesse metriche** — ordinamento (Gini, PR-AUC, KS), probabilita' (Brier) e **calibrazione per fasce** col test di Jeffreys. Come challenger scegliamo la rete calibrata col metodo migliore in tabella (tipicamente Platt o isotonica).

In [ ]:
# --- Tabella finale champion/challenger -----------------------------------------------
migliore_cal = min(["Platt", "isotonica", f"temperatura (T={T_best:.1f})"],
                   key=lambda k: brier_score_loss(y_test, candidati[k]))
proba_rete_cal = candidati[migliore_cal]
print(f"Challenger: rete con embedding + calibrazione '{migliore_cal}'\n")

finale = pd.DataFrame({
    "Logistica":            valuta_modello(y_test, proba_log_test),
    "XGBoost (champion)":   valuta_modello(y_test, proba_xgb_test),
    "Rete+emb. calibrata":  valuta_modello(y_test, proba_rete_cal),
}).T.round(4)
print(finale[["Gini", "PR-AUC", "KS", "Brier", "recall", "accuracy"]].to_string())
print()
for nome, p in [("Logistica   ", proba_log_test), ("XGBoost     ", proba_xgb_test),
                ("Rete calibr.", proba_rete_cal)]:
    g = 2 * roc_auc_score(y_test, p) - 1
    print(f"Gini {nome}: {g:.1%}  ->  {semaforo_gini(g)}")

In [ ]:
# --- Calibrazione per fasce, come la legge un validatore -------------------------------
print("CHALLENGER (rete calibrata) - fasce di rating e test di Jeffreys:")
print(tabella_calibrazione(y_test, proba_rete_cal, n_fasce=8).round(4).to_string())
print("\nCHAMPION (XGBoost) - stesse fasce, stesso test:")
print(tabella_calibrazione(y_test, proba_xgb_test, n_fasce=8).round(4).to_string())

**Come leggere il verdetto.**

- Su questo tabellare *application-level* il champion resta tipicamente **davanti o alla pari** di misura: e' il risultato atteso dalla letteratura di benchmark (Grinsztajn 2022 e colleghi) e **non e' una sconfitta** — il challenger di ieri perdeva nettamente, quello di oggi gioca la stessa partita con in piu' la calibrazione sotto controllo;
- il valore vero costruito oggi e' il **metodo**: pipeline senza leakage, embedding per l'alta cardinalita', esperimenti controllati, calibrazione verificata per fasce. E' la *dotazione minima* per difendere un modello davanti alla Vigilanza;
- gli **embedding** sono il ponte verso domani: nella mattina 3 le stesse lookup table rappresenteranno le **causali delle transazioni** dentro le sequenze — il territorio dove il champion non puo' seguirci, perche' un albero non legge l'ordine temporale.

**Coda della pillola 3 — i tre regimi dei dati (e il low-default).** Dove ha senso tutto questo, portafoglio per portafoglio? **Retail big data** (milioni di righe): il terreno naturale di embedding e reti. **SME**: regime intermedio, decide il confronto misurato. **Large corporate low-default**: con — poniamo — 500 controparti e 10 default, *qualunque* Gini puntuale balla di ±10–15 punti tra un campione e l'altro, i test per fascia non hanno potenza, e il problema non e' «rete o alberi» ma **l'incertezza delle stime**: intervalli di confidenza (DeLong, bootstrap), prudenza alla Pluto–Tasche, pooling. Su quei portafogli il deep learning non e' la risposta; saperlo dire e' parte della professionalita' del validatore.

---
# Sintesi della mattina

1. **Preprocessing per reti**: imputazione + flag di mancanza, scaling adattato sul solo train (il leakage si evita per costruzione), categoriche in indici con lo 0 riservato ai livelli mai visti;
2. **embedding**: le categoriche ad alta cardinalita' diventano vettori densi appresi — `nn.Embedding` e' una lookup table che l'addestramento riempie di significato;
3. **regolarizzazione con metodo**: dropout, batch norm, weight decay ed early stopping, misurati con esperimenti controllati — una tecnica alla volta;
4. **sbilanciamento**: `pos_weight` e focal loss cambiano recall e probabilita', quasi mai l'ordinamento; se si riscala la loss, **si ricalibra**;
5. **calibrazione**: temperatura, Platt, isotonica — adattate su validation, giudicate sul test, lette per fasce col test di Jeffreys;
6. il **confronto champion/challenger** ora e' completo e difendibile: stessi dati, stesse metriche, calibrazione verificata.

---
# Esercizi — a difficolta' crescente

Celle predisposte qui sotto; le **soluzioni** arrivano a fine corso. Gli esercizi 4, 5 e 6 sono opzionali, per chi vuole spingersi oltre.

**E1 (guidato) — La dimensione degli embedding.** Sostituire la regola pratica con dimensioni fisse: 2 per tutte, poi 32 per tutte (modificare `dimensione_embedding` o passare una variante della rete). Riaddestrare la configurazione base e confrontare Gini di validation e numero di parametri. Dove smette di rendere?

**E2 (guidato) — Che cosa ha imparato l'embedding?** Estrarre la matrice dell'embedding di `ORGANIZATION_TYPE` dalla rete finale (`rete_finale.embedding[...]`), calcolare la **similarita' coseno** tra il vettore di «Bank» e tutti gli altri livelli, stampare i 5 piu' vicini e i 5 piu' lontani. I vicini hanno senso economico?

**E3 — La griglia del dropout.** Riaddestrare con `p_dropout` in {0,1, 0,3, 0,5} a parita' di tutto il resto. Quale vince sul Gini di validation? Le curve di apprendimento mostrano il compromesso stabilita'/velocita'?

**E4 (opzionale) — Focal a parita' di budget.** Confrontare focal loss con γ in {1, 2, 3} contro `pos_weight`, tutte con lo **stesso numero massimo di epoche e la stessa pazienza**: a parita' di budget di addestramento, chi ordina meglio? E chi stima la PD media piu' vicina al vero?

**E5 (opzionale) — Calibrare anche il champion.** Applicare la calibrazione isotonica **a XGBoost** (adattata su validation) e rifare la tabella finale: il Brier del champion migliora? E il test di Jeffreys per fasce?

**E6 (opzionale) — Random Forest nel confronto.** Aggiungere una `RandomForestClassifier` (scikit-learn) alla tabella champion/challenger: stesso preprocessing di XGBoost (encoding ordinale; attenzione — la foresta non gestisce i NaN: servira' l'imputazione), stesse metriche, stesso test. Dove si piazza tra logistica, XGBoost e rete?

In [ ]:
# === E1 - dimensione degli embedding =========================================
# TODO: variante con dimensioni fisse (2, poi 32), riaddestrare, confrontare
#       Gini val e numero di parametri


In [ ]:
# === E2 - leggere l'embedding di ORGANIZATION_TYPE ===========================
# TODO: estrarre la matrice dei pesi, similarita' coseno col vettore di 'Bank',
#       stampare i 5 livelli piu' vicini e i 5 piu' lontani
# Suggerimento: l'indice di 'Bank' e' mappe['ORGANIZATION_TYPE']['Bank'];
#               la matrice e' rete_finale.embedding[j].weight (j = posizione
#               di ORGANIZATION_TYPE in COL_CATEGORICHE)


In [ ]:
# === E3 - griglia del dropout ================================================
# TODO: ciclo su p_dropout in {0.1, 0.3, 0.5}, tabella Gini val + curve


In [ ]:
# === E4 (opzionale) - focal vs pos_weight a parita' di budget ================
# TODO: gamma in {1, 2, 3} e pos_weight, stesse epoche e pazienza;
#       confrontare Gini, PR-AUC e PD media stimata


In [ ]:
# === E5 (opzionale) - calibrare il champion ==================================
# TODO: IsotonicRegression su (proba_xgb_val, y_val), applicare al test,
#       rifare tabella finale e tabella_calibrazione


In [ ]:
# === E6 (opzionale) - Random Forest nel confronto ============================
# TODO: RandomForestClassifier con encoding ordinale + imputazione,
#       valuta_modello sul test, aggiungere la riga alla tabella finale


---
*Riferimenti.* Dataset: *Home Credit Default Risk*, Kaggle competition (2018), dati © Home Credit Group — campione fornito col materiale del corso. Embedding per categoriche: Guo, C., Berkhahn, F. (2016), «Entity Embeddings of Categorical Variables», arXiv:1604.06737. Focal loss: Lin, T.-Y. et al. (2017), «Focal Loss for Dense Object Detection», ICCV. Calibrazione delle reti: Guo, C. et al. (2017), «On Calibration of Modern Neural Networks», ICML (temperature scaling); Platt, J. (1999); Zadrozny, B., Elkan, C. (2002) (isotonica). Benchmark tabellare: Grinsztajn, L., Oyallon, E., Varoquaux, G. (2022), «Why do tree-based models still outperform deep learning on typical tabular data?», NeurIPS Datasets and Benchmarks. Le soglie e i test citati seguono le ipotesi di lavoro del corso (quadro di riferimento, I1–I3), dichiarate come prassi e in attesa di validazione da parte della funzione di validazione interna.

*Prossimo appuntamento: Mattina 3 — Sequenze di transazioni: dove il deep learning vince.*